# Pruebas clasificacion de olas

In [0]:
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.cluster import DBSCAN, k_means
import itertools
from pyspark.sql import functions as F
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [0]:
ambiente = 'project'
PORCENTAJE_ENTRENAMIENTO = 0.1
costa = 'Salina Cruz'

## Seleccionar los datos

In [0]:
data = (
    spark.sql(
        f"""
            SELECT *
            FROM cor_{ambiente}.silver.swell_metrics
            WHERE coast_name IN ('{costa}')
        """
    )
)

In [0]:
data_pre_processing = (
    data
    .withColumn('coast_year_month', F.concat(F.col('coast_name'), F.lit('_'), F.date_format('datetime', 'yyyy-MM')))
)

coast_year_month_dict = {row.coast_year_month: PORCENTAJE_ENTRENAMIENTO for row in data_pre_processing.select('coast_year_month').distinct().collect()}

data_sample = (
    data_pre_processing
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=0)
    .drop('coast_year_month')
).toPandas()

coast_names = data_sample['coast_name'].unique()

## Preparar los datos

In [0]:
features = ['wind_u', 'wind_v', 'wave_u', 'wave_v', 'wave_period_s']

In [0]:
if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'
else:
    base_path = Path.cwd().parent
    scaler_path = f'{base_path}/scaler/scaler_{{}}.pkl'

In [0]:
X_list = []

for coast in coast_names[:1]:
    scaler_path_coast = scaler_path.format(coast)
    scaler = joblib.load(scaler_path_coast)

    df_coast = data_sample[data_sample['coast_name'] == coast].copy()

    X_coast_scaled = scaler.transform(df_coast[features])

    X_coast_scaled_df = pd.DataFrame(X_coast_scaled, columns=features)
    X_coast_scaled_df.columns = [f'{col}_scaled' for col in X_coast_scaled_df.columns]
    df_coast = pd.concat([df_coast, X_coast_scaled_df], axis=1)

    X_list.append(df_coast)

X = pd.concat(X_list, ignore_index=True)

# Graficar sin clusters

In [0]:
fig =px.scatter(
    x=X['wind_speed_ms'],
    y=X['wave_height_m'],
    color=X['wave_period_s'],
    title=f'Wind Speed vs Wave Height {costa}',
    labels={
        'x': 'Wind Speed (m/s)',
        'y': 'Wave Height (m)',
        'color': 'Wave Period (s)'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.write_html(f'wind_speed_wave_height_{costa.lower().replace(" ", "_")}.html')
fig.show()

In [0]:
fig =px.scatter(
    x=X['wave_period_s'],
    y=X['wave_height_m'],
    color=X['wind_speed_ms'],
    color_discrete_sequence=px.colors.qualitative.Pastel1,
    title=f'Wave period vs Wave Height {costa}',
    labels={
        'x': 'Wave Period (s)',
        'y': 'Wave Height (m)',
        'color': 'Wind Speed (m/s)'
    }
)
fig.update_layout(
    legend=dict(
        orientation="h",
        y=-0.2
    )
)
fig.update_xaxes(range=[0,18])
fig.update_yaxes(range=[0,8])
fig.write_html(f'{os.getcwd()}/wave_period_vs_wave_height_{costa.lower().replace(" ", "_")}.html')
fig.show()

In [0]:
fig =px.scatter(
    x=X['wave_period_s'],
    y=X['wind_speed_ms'],
    color=X['wave_height_m'],
    color_discrete_sequence=px.colors.qualitative.Pastel1,
    title=f'Wave period vs Wind speed {costa}',
    labels={
        'x': 'Wave Period (s)',
        'y': 'Wind Speed (m/s)',
        'color': 'Wave Height (m)'
    }
)
fig.update_layout(
    legend=dict(
        orientation="h",
        y=-0.2
    )
)
fig.update_xaxes(range=[0,18])
fig.update_yaxes(range=[0,22])
fig.write_html(f'{os.getcwd()}/wave_period_vs_wind_speed_{costa.lower().replace(" ", "_")}.html')
fig.show()

# DBSCAN

## Preparar pruebas de parametros

In [0]:
posible_e = np.linspace(0.01, 1, 25)
min_samples_range = 10
max_samples_range = 50
posible_min_samples = np.arange(
    min_samples_range,
    max_samples_range,
    1
)
combinaciones = list(itertools.product(posible_e, posible_min_samples))

min_pct_huracanes = X.shape[0] * 0.005
max_pct_huracanes = X.shape[0] * 0.05
print(len(combinaciones))

## Evaluar modelos

In [0]:
# https://www.youtube.com/watch?v=VO_uzCU_nKw
def get_scores_and_labels(combinations, X):
  parametros_ok = []
  for i, (eps, num_samples) in enumerate(combinations):
    dbscan_cluster_model = DBSCAN(eps=eps, min_samples=num_samples).fit(X)
    labels = dbscan_cluster_model.labels_
    labels_set = set(labels)
    num_clusters = len(labels_set)
    if -1 in labels_set:
      num_clusters -= 1
    else:
      continue
      
    if (num_clusters < 1):
      continue
    clusters_conteo = dict(zip(*np.unique(labels, return_counts=True)))
    if (clusters_conteo.get(-1) > max_pct_huracanes) or (clusters_conteo.get(-1) < min_pct_huracanes):
      continue
    
    parametros_ok.append({
        'eps': eps,
        'num_samples': num_samples,
        'labels': labels,
        'clusters_conteo': clusters_conteo
    })

  return parametros_ok

parametros_ok = get_scores_and_labels(combinaciones, X[features])

In [0]:
print(f'Número de combinaciones que cumplen con el porcentaje de huracanes: {len(parametros_ok)}')

In [0]:
for p in parametros_ok:
    cluster, conteos = np.unique(p['labels'], return_counts=True)
    print(f"eps: {p['eps']}, num_samples: {p['num_samples']}")
    print(p['clusters_conteo'])

In [0]:
total_puntos = X.shape[0]
px.scatter(
    x=[p['eps'] for p in parametros_ok],
    y=[p['num_samples'] for p in parametros_ok],
    color=[(100*p['clusters_conteo'].get(-1)/total_puntos) for p in parametros_ok],
    title='DBSCAN Parameters',
    hover_data={'index': list(range(len(parametros_ok)))},
    labels={
        'x': 'Epsilon',
        'y': 'Min Samples',
        'color': 'Noise Points (%)',
        'index': 'id'
    }
)

## Asignar labels de DBSCAN a los datos

In [0]:
best_index = 120
X['cluster'] = parametros_ok[best_index]['labels'].astype(str)

## Graficar clusters

In [0]:
fig =px.scatter(
    x=X['wind_speed_ms'],
    y=X['wave_height_m'],
    color=parametros_ok[best_index]['labels'].astype(str),
    title=f'DBSCAN cluster: Wind Speed vs Wave Height {costa}',
    labels={
        'x': 'Wind Speed (m/s)',
        'y': 'Wave Height (m)',
        'color': 'Cluster'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.update_xaxes(range=[-1,22])
fig.update_yaxes(range=[-1,6])
fig.write_html(f'dbscan_clusters_wind_speed_wave_height_{costa.lower().replace(" ", "_")}.html')
fig.show()

In [0]:
fig =px.scatter(
    x=X['wave_period_s'],
    y=X['wave_height_m'],
    color=parametros_ok[best_index]['labels'].astype(str),
    title=f'DBSCAN cluster: Wave Period vs Wave Height {costa}',
    labels={
        'x': 'Wave Period (s)',
        'y': 'Wave Height (m)',
        'color': 'Cluster'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.update_xaxes(range=[3,16])
fig.update_yaxes(range=[-1,6])
fig.write_html(f'dbscan_clusters_wave_period_wave_height_{costa.lower().replace(" ", "_")}.html')
fig.show()

In [0]:
fig =px.scatter(
    x=X['wind_speed_ms'],
    y=X['wave_period_s'],
    color=parametros_ok[best_index]['labels'].astype(str),
    title=f'DBSCAN cluster: Wind Speed vs Wave Period {costa}',
    labels={
        'x': 'Wind Speed (m/s)',
        'y': 'Wave Period (s)',
        'color': 'Cluster'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.update_xaxes(range=[-1,22])
fig.update_yaxes(range=[3,16])
fig.write_html(f'dbscan_clusters_wind_speed_wave_period_{costa.lower().replace(" ", "_")}.html')
fig.show()

# K-means

## Parametros

In [0]:
n_clusters = 5

## Modelo

In [0]:
kmeans = k_means(X[X['cluster'] != '-1'][features], n_clusters=n_clusters)
kmeans_labels = kmeans[1].astype(str)

## Asignar labels de K-means a los datos

In [0]:
X.loc[X['cluster'] != '-1', 'cluster'] = kmeans_labels

## Graficar clusters

In [0]:
fig =px.scatter(
    x=X[X['cluster'] != '-1']['wind_speed_ms'],
    y=X[X['cluster'] != '-1']['wave_height_m'],
    color=kmeans_labels,
    title=f'K-Means cluster: Wind Speed vs Wave Height {costa}',
    labels={
        'x': 'Wind Speed (m/s)',
        'y': 'Wave Height (m)',
        'color': 'Cluster'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.update_xaxes(range=[-1,22])
fig.update_yaxes(range=[-1,6])
fig.write_html(f'kmeans_clusters_wind_speed_wave_height_{costa.lower().replace(" ", "_")}.html')
fig.show()

In [0]:
fig =px.scatter(
    x=X[X['cluster'] != '-1']['wave_period_s'],
    y=X[X['cluster'] != '-1']['wave_height_m'],
    color=kmeans_labels,
    title=f'K-Means cluster: Wave Period vs Wave Height {costa}',
    labels={
        'x': 'Wave Period (s)',
        'y': 'Wave Height (m)',
        'color': 'Cluster'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.update_xaxes(range=[3,16])
fig.update_yaxes(range=[-1,6])
fig.write_html(f'kmeans_clusters_wave_period_wave_height_{costa.lower().replace(" ", "_")}.html')
fig.show()

In [0]:
fig =px.scatter(
    x=X[X['cluster'] != '-1']['wind_speed_ms'],
    y=X[X['cluster'] != '-1']['wave_period_s'],
    color=kmeans_labels,
    title=f'K-Means cluster: Wind Speed vs Wave Period {costa}',
    labels={
        'x': 'Wind Speed (m/s)',
        'y': 'Wave Period (s)',
        'color': 'Cluster'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.update_xaxes(range=[-1,22])
fig.update_yaxes(range=[3,16])
fig.write_html(f'kmeans_clusters_wind_speed_wave_period_{costa.lower().replace(" ", "_")}.html')
fig.show()

# Graficar clusters con DBSCAN y K-means juntos

In [0]:
fig =px.scatter(
    x=X['wind_speed_ms'],
    y=X['wave_height_m'],
    color=X['cluster'],
    title=f'K-Means cluster: Wind Speed vs Wave Height {costa}',
    labels={
        'x': 'Wind Speed (m/s)',
        'y': 'Wave Height (m)',
        'color': 'Cluster'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.update_xaxes(range=[-1,22])
fig.update_yaxes(range=[-1,6])
fig.write_html(f'clusters_wind_speed_wave_height_{costa.lower().replace(" ", "_")}.html')
fig.show()

In [0]:
fig =px.scatter(
    x=X['wave_period_s'],
    y=X['wave_height_m'],
    color=X['cluster'],
    title=f'K-Means cluster: Wave Period vs Wave Height {costa}',
    labels={
        'x': 'Wave Period (s)',
        'y': 'Wave Height (m)',
        'color': 'Cluster'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.update_xaxes(range=[3,16])
fig.update_yaxes(range=[-1,6])
fig.write_html(f'clusters_wave_period_wave_height_{costa.lower().replace(" ", "_")}.html')
fig.show()

In [0]:
fig =px.scatter(
    x=X['wind_speed_ms'],
    y=X['wave_period_s'],
    color=X['cluster'],
    title=f'K-Means cluster: Wind Speed vs Wave Period {costa}',
    labels={
        'x': 'Wind Speed (m/s)',
        'y': 'Wave Period (s)',
        'color': 'Cluster'
    }
)
fig.update_layout(
    legend=dict(
        orientation="v",
        y=1,
        x=1.02
    )
)
fig.update_xaxes(range=[-1,22])
fig.update_yaxes(range=[3,16])
fig.write_html(f'clusters_wind_speed_wave_period_{costa.lower().replace(" ", "_")}.html')
fig.show()

# Describir caracteristicas de cada cluster

In [0]:
clusters = range(-1, n_clusters)
features = ['wind_speed_ms', 'wave_height_m', 'wave_period_s']
titulos = [f'cluster {i[0]} ({i[1]})' for i in itertools.product(clusters, features)]
ejes = [{'min': 0, 'max': 0} for _ in features]
fig = make_subplots(
    rows=len(clusters), 
    cols=len(features), 
    subplot_titles=titulos,
    vertical_spacing=0.05
)
for cluster in clusters:
    for feature in features:
        fig.add_trace(
            go.Box(
                x=X[X['cluster'] == str(cluster)][feature],
                orientation='h'
            ),
            row=cluster+2, col=features.index(feature)+1
        )
        ejes[features.index(feature)]['max'] = max(ejes[features.index(feature)]['max'], X[X['cluster'] == str(cluster)][feature].max())
        fig.update_yaxes(showticklabels=False, row=cluster+2, col=features.index(feature)+1)
        fig.update_xaxes(range=[ejes[features.index(feature)]['min'], ejes[features.index(feature)]['max']*1.1], row=cluster+2, col=features.index(feature)+1)
fig.update_layout(
    height=n_clusters*250,
    showlegend=False,
    margin=dict(t=100, b=50, l=50, r=50)
)